In [0]:
# %python
# ============================================================
# REST API demo
#
# Triggers medallion-orchestration-job with a specific run_date.
# Auth is injected automatically inside Databricks workspace.
# In external systems, replace token with a service principal PAT.
# ============================================================

import requests
import json
import time

host  = "https://dbc-57fd889f-05d5.cloud.databricks.com"
token = "YOUR_TOKEN"

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

In [0]:
# %python
# List jobs and find medallion-orchestration-job
response = requests.get(
    f"{host}/api/2.1/jobs/list",
    headers=headers,
    params={"name": "medallion-orchestration-job"}
)

data = response.json()
job_id = data["jobs"][0]["job_id"]
print(f"\nJob ID: {job_id}")

In [0]:
# %python
# Trigger a job run — equivalent to "Run now with different parameters"
payload = {
    "job_id": job_id,
    "job_parameters": {
        "run_date": "2024-03-01"
    }
}

response = requests.post(
    f"{host}/api/2.1/jobs/run-now",
    headers=headers,
    json=payload
)

run_data = response.json()
run_id = run_data["run_id"]
print(f"\nRun ID: {run_id}")

In [0]:
# %python
# Poll for run completion
# In production: use a webhook instead of polling —
# configure webhook_notifications.on_success / on_failure
# on the job to receive a callback when the run finishes.
# Polling is fine for demos and simple scripts.

print(f"Polling run {run_id}...")

while True:
    response = requests.get(
        f"{host}/api/2.1/jobs/runs/get",
        headers=headers,
        params={"run_id": run_id}
    )
    run = response.json()
    state = run["state"]["life_cycle_state"]
    print(f"  State: {state}")

    if state in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
        result = run["state"].get("result_state", "UNKNOWN")
        print(f"\nRun completed. Result: {result}")
        break

    time.sleep(15)

####CLI Commands



```bash

 databricks --version

 databricks jobs list --name "medallion-orchestration-job"

 databricks jobs run-now --json '{"job_id": 119866108986758, "job_parameters": {"run_date": "2025-03-02"}}'



```